## 📖 [The RAG Techniques Book is HERE](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--choose-chunk-size&click=book-buy-amazon&target=https%3A%2F%2Fwww.amazon.com%2Fdp%2FB0D76734SZ%3Ftag%3Ddiamantai-ragnb-20&text=)

**The super extended version of this repository.** The book goes far beyond the notebooks: the **intuition** behind every technique, **side-by-side comparisons** showing when each approach wins (and when it quietly fails), and **illustrations** that make the tricky parts finally click.

⏳ **Kindle $9.99 / Paperback $24.99 / Free with Kindle Unlimited.** An Amazon Bestseller in Generative AI (hit #1 in Generative AI on Amazon at launch).

### 👉 [Get the book on Amazon](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--choose-chunk-size&click=book-buy-amazon&target=https%3A%2F%2Fwww.amazon.com%2Fdp%2FB0D76734SZ%3Ftag%3Ddiamantai-ragnb-20&text=)

---


In [27]:
from IPython.display import IFrame

# 显示本地 HTML 文件
IFrame(src='../Flowchart_explanation/choose_chunk_size.html', width='100%', height=600)

# Package Installation and Imports

The cell below installs all necessary packages required to run this notebook.


In [1]:
# Install required packages
# !pip install llama-index

  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached sqlalchemy-2.0.50-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.9 MB ? eta -:--:--
    --------------------------------------- 0.3/11.9 MB ? eta -:--:--
    --------------------------------------- 0.3/11.9 MB ? eta -:--:--
    --------------------------------------- 0.3/11.9 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.9 MB 479.9 kB/s eta 0:00:24
   - -------------------------------------- 0.5/11.9 MB 479.9 kB/s eta 0:00:24
   -- ------------------------------------- 0.8/11.9 MB 562.7 kB/s eta 0:00:20
   -- ------------------------------------- 0.8/11.9 MB 562.7 kB/s eta 0:00:20
   --- ------------------------------------ 1.0/11.9 MB 568


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import nest_asyncio
import random

nest_asyncio.apply()
from dotenv import load_dotenv

from llama_index.core import VectorStoreIndex, SimpleDirectoryReader
from llama_index.core.prompts import PromptTemplate

from llama_index.core.evaluation import (
    DatasetGenerator,
    FaithfulnessEvaluator,
    RelevancyEvaluator
)

from llama_index.core import Settings

import time
import os



### Read Docs

In [2]:
data_dir = "../data/minidata"
documents = SimpleDirectoryReader(data_dir).load_data()
print(f"加载了 {len(documents)} 个文档片段")

加载了 1 个文档片段


In [3]:
#使用qwen-max生成所有分片的问题
model = 'qwen-max'
from llama_index.llms.openai_like import OpenAILike
llm = OpenAILike(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model=model,  # 明确指定模型
    temperature=0.7,
    is_chat_model=True,  # 这个参数很重要
)

### Create evaluation questions and pick k out of them

In [4]:
num_eval_questions = 25

eval_documents = documents[0:20]

#如果文档很多，或者文档很大，分片生成问题超级慢
#默认分片1024
data_generator = DatasetGenerator.from_documents(
    eval_documents,
    llm=llm
)
eval_questions = data_generator.generate_questions_from_nodes()
print(f"生成了 {len(eval_questions)} 个问题")
print(f"前5个问题: {eval_questions[:5]}")

#随机获取25个问题
k_eval_questions = random.sample(eval_questions, num_eval_questions)

D:\Desktop\RAG_Techniques\RAG_Techniques\.venv\Lib\site-packages\llama_index\core\evaluation\dataset_generation.py:201: DeprecationWarning: Call to deprecated class DatasetGenerator. (Deprecated in favor of `RagDatasetGenerator` which should be used instead.)
  return cls(


生成了 1838 个问题
前5个问题: ['Certainly! Here are 10 diverse questions based on the context information provided:', 'What is the file path of the PDF document mentioned in the context?', 'What is the size of the "Understanding_Climate_Change.pdf" file in bytes?', 'On which date was the "Understanding_Climate_Change.pdf" file last modified?', 'What is the type of the file "Understanding_Climate_Change.pdf"?']


D:\Desktop\RAG_Techniques\RAG_Techniques\.venv\Lib\site-packages\llama_index\core\evaluation\dataset_generation.py:297: DeprecationWarning: Call to deprecated class QueryResponseDataset. (Deprecated in favor of `LabelledRagDataset` which should be used instead.)
  return QueryResponseDataset(queries=queries, responses=responses_dict)


### Define metrics evaluators and modify llama_index faithfullness evaluator prompt to rely on the context 

In [10]:
#使用其他模型来基准检测response的相关性和忠实度
model = 'qwen3-max'
llm = OpenAILike(
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
    model=model,  # 明确指定模型
    temperature=0,  #完全确定性输出，每次相同输入得到相同结果，适合评估任务
    is_chat_model=True,  # 这个参数很重要
)

# Set appropriate settings for the LLM
Settings.llm = llm

# Define Faithfulness Evaluators which are based on GPT-4
faithfulness_qwen = FaithfulnessEvaluator()

faithfulness_new_prompt_template = PromptTemplate(""" Please tell if a given piece of information is directly supported by the context.
    You need to answer with either YES or NO.
    Answer YES if any part of the context explicitly supports the information, even if most of the context is unrelated. If the context does not explicitly support the information, answer NO. Some examples are provided below.

    Information: Apple pie is generally double-crusted.
    Context: An apple pie is a fruit pie in which the principal filling ingredient is apples.
    Apple pie is often served with whipped cream, ice cream ('apple pie à la mode'), custard, or cheddar cheese.
    It is generally double-crusted, with pastry both above and below the filling; the upper crust may be solid or latticed (woven of crosswise strips).
    Answer: YES

    Information: Apple pies taste bad.
    Context: An apple pie is a fruit pie in which the principal filling ingredient is apples.
    Apple pie is often served with whipped cream, ice cream ('apple pie à la mode'), custard, or cheddar cheese.
    It is generally double-crusted, with pastry both above and below the filling; the upper crust may be solid or latticed (woven of crosswise strips).
    Answer: NO

    Information: Paris is the capital of France.
    Context: This document describes a day trip in Paris. You will visit famous landmarks like the Eiffel Tower, the Louvre Museum, and Notre-Dame Cathedral.
    Answer: NO

    Information: {query_str}
    Context: {context_str}
    Answer:

    """)

faithfulness_qwen.update_prompts({"your_prompt_key": faithfulness_new_prompt_template}) # Update the prompts dictionary with the new prompt template

# Define Relevancy Evaluators which are based on GPT-4
relevancy_qwen = RelevancyEvaluator()

### Function to evaluate metrics for each chunk size

In [22]:
# Define function to calculate average response time, average faithfulness and average relevancy metrics for given chunk size
# We use GPT-3.5-Turbo to generate response and GPT-4 to evaluate it.

from llama_index.embeddings.dashscope import DashScopeEmbedding
def evaluate_response_time_and_accuracy(chunk_size, eval_questions):
    """
    Evaluate the average response time, faithfulness, and relevancy of responses generated by GPT-3.5-turbo for a given chunk size.
    
    Parameters:
    chunk_size (int): The size of data chunks being processed.
    
    Returns:
    tuple: A tuple containing the average response time, faithfulness, and relevancy metrics.
    """

    total_response_time = 0
    total_faithfulness = 0
    total_relevancy = 0

    # 配置 Embedding 模型（使用通义千问的兼容接口）
    # 先安装依赖：pip install llama-index-embeddings-dashscope
    embed_model = DashScopeEmbedding(
        model_name="text-embedding-v2",  # 或 text-embedding-v1
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        max_batch_size=20,  # 关键：设置批次大小不超过10
        timeout=60.0,
    )

    # create vector index
    model ='qwen-turbo'
    llm = OpenAILike(
        api_key=os.getenv("DASHSCOPE_API_KEY"),
        api_base="https://dashscope.aliyuncs.com/compatible-mode/v1",
        model=model,  # 明确指定模型
        temperature=0.7,  #多样性输出
        is_chat_model=True,  # 这个参数很重要
        timeout=60.0,
    )

    Settings.llm = llm
    Settings.embed_model = embed_model
    Settings.chunk_size = chunk_size
    Settings.chunk_overlap = chunk_size // 5 

    print("ready for spilitting")
    vector_index = VectorStoreIndex.from_documents(eval_documents)

    
    # build query engine
    query_engine = vector_index.as_query_engine(similarity_top_k=5)
    num_questions = len(eval_questions)
    print("split ok")

    # Iterate over each question in eval_questions to compute metrics.
    # While BatchEvalRunner can be used for faster evaluations (see: https://docs.llamaindex.ai/en/latest/examples/evaluation/batch_eval.html),
    # we're using a loop here to specifically measure response time for different chunk sizes.
    for question in eval_questions:
        start_time = time.time()
        response_vector = query_engine.query(question)
        elapsed_time = time.time() - start_time
        
        faithfulness_result = faithfulness_qwen.evaluate_response(
            response=response_vector
        ).passing
        
        relevancy_result = relevancy_qwen.evaluate_response(
            query=question, response=response_vector
        ).passing

        total_response_time += elapsed_time
        total_faithfulness += faithfulness_result
        total_relevancy += relevancy_result

    average_response_time = total_response_time / num_questions
    average_faithfulness = total_faithfulness / num_questions
    average_relevancy = total_relevancy / num_questions

    return average_response_time, average_faithfulness, average_relevancy

### Test different chunk sizes 

In [23]:
chunk_sizes = [128, 256]

for chunk_size in chunk_sizes:
  avg_response_time, avg_faithfulness, avg_relevancy = evaluate_response_time_and_accuracy(chunk_size, k_eval_questions)
  print(f"Chunk size {chunk_size} - Average Response time: {avg_response_time:.2f}s, Average Faithfulness: {avg_faithfulness:.2f}, Average Relevancy: {avg_relevancy:.2f}")

ready for spilitting
split ok
Chunk size 128 - Average Response time: 1.10s, Average Faithfulness: 0.60, Average Relevancy: 0.72
ready for spilitting
split ok
Chunk size 256 - Average Response time: 1.03s, Average Faithfulness: 0.64, Average Relevancy: 0.80


![](https://europe-west1-rag-techniques-views-tracker.cloudfunctions.net/rag-techniques-tracker?notebook=all-rag-techniques--choose-chunk-size)